# Meta-Signal — Fine-Tuned Model Evaluation

Evaluates `Anvit25/meta-signal-q4-agent` against the live HF Space API.
Runs 3 seeds × Tasks 5/6/7 = 9 episodes, then compares against ExpertBot baseline.

**Runtime**: ~25-40 min on A10G (9 episodes, up to 100 steps each + inference per step)  
**Requires**: GPU (A10G Small), HF write token, HF Space `Anvit25/meta-signal` running

In [ ]:
# -- 1. Install ----------------------------------------------------------------
!pip install --quiet unsloth
!pip install --quiet transformers accelerate bitsandbytes peft huggingface_hub requests
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# -- 2. Config -----------------------------------------------------------------
import os

HF_TOKEN   = ""   # paste your HF token here (needs read scope)
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

SPACE_URL  = "https://anvit25-meta-signal.hf.space"   # live environment API
MODEL_REPO = "Anvit25/meta-signal-q4-agent"           # fine-tuned LoRA adapter

EVAL_TASKS = [5, 6, 7]
SEEDS      = [42, 123, 456]   # 3 seeds per task -> 9 episodes total

BASELINE   = {5: 0.716, 6: 0.542, 7: 0.663}   # ExpertBot scores (seed=42)
TASK_NAMES = {5: "Signal Recovery", 6: "Andromeda Stability", 7: "Q4 Champion"}
TASK_STEPS = {5: 30, 6: 75, 7: 100}

print(f"Tasks: {EVAL_TASKS}  Seeds: {SEEDS}  -> {len(EVAL_TASKS)*len(SEEDS)} episodes total")

In [ ]:
# -- 3. Load fine-tuned model --------------------------------------------------
from huggingface_hub import login
from unsloth import FastLanguageModel

login(token=HF_TOKEN)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print(f"Loaded: {MODEL_REPO}")

In [ ]:
# -- 4. Helpers ----------------------------------------------------------------
import json, re, requests

ALPACA_PROMPT = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PHASE_INSTRUCTIONS = {
    "Nominal": (
        "You are an advertising budget optimisation agent in Phase 1 (days 1-20). "
        "Signal is clean. Identify the best campaign and concentrate budget there "
        "while staying below 70% concentration. "
        "Output a JSON action with allocations, attribution, feature_mask, use_capi, pacing_speed."
    ),
    "Signal_Loss": (
        "You are an advertising budget optimisation agent in Phase 2 (days 21-50). "
        "iOS ATT has caused 3x noise. Set use_capi=true for clean counts (costs 2.0 epsilon). "
        "Ration CAPI to 1 call per 3-5 steps. Between calls hold your Phase 1 allocation steady. "
        "Output a JSON action with allocations, attribution, feature_mask, use_capi, pacing_speed."
    ),
    "Andromeda_Glitched": (
        "You are an advertising budget optimisation agent in Phase 3 (days 51-80). "
        "Any allocation change over 20% of total budget triggers a 7-day CVR reset. "
        "Freeze your allocation. If learning_status is Reset do not change allocations. "
        "Output a JSON action with allocations, attribution, feature_mask, use_capi, pacing_speed."
    ),
    "Peak_Load": (
        "You are an advertising budget optimisation agent in Phase 4 (days 81-100). "
        "pacing_speed above 1.5 triggers 30% overspend risk per step. "
        "Set pacing_speed=1.0 and hold your Phase 3 allocation. Survive to episode end. "
        "Output a JSON action with allocations, attribution, feature_mask, use_capi, pacing_speed."
    ),
}

CAMPAIGN_NAMES = ["camp_feed", "camp_reels", "camp_stories"]
VALID_FEATURES = [f"I{i}" for i in range(1, 14)] + [f"C{i}" for i in range(1, 27)]


def obs_to_text(obs):
    """Convert API observation dict to the Alpaca input format used during training."""
    camps = []
    for cs in obs.get("campaigns", []):
        ci = cs.get("confidence_interval", [0, 0])
        camps.append(
            f"{cs['campaign_id']}: spend=${cs.get('spend', 0):.1f}, "
            f"noisy_conv={cs.get('noisy_conversions', 0):.2f}, "
            f"est_roas={cs.get('estimated_roas', 0):.3f}, "
            f"ctr={cs.get('ctr', 0):.4f}, "
            f"ci=[{ci[0]:.2f},{ci[1]:.2f}]"
        )
    lines = [
        f"step={obs.get('step',0)} day={obs.get('day',0)} phase={obs.get('platform_health','Nominal')}",
        f"budget_remaining=${obs.get('total_budget_remaining',0):.2f} epsilon={obs.get('epsilon_remaining',0):.3f}",
        f"privacy_regime={obs.get('privacy_regime','standard')} learning_status={obs.get('learning_status','Optimized')}",
        f"market_trend={obs.get('market_trend','Rising')} regulatory_violation={obs.get('regulatory_violation',False)}",
        f"campaigns: {' | '.join(camps)}",
    ]
    if obs.get("warning"):
        lines.append(f"warning: {obs['warning']}")
    return "\n".join(lines)


def sanitize_action(data, obs):
    """Clamp allocations to budget, filter feature_mask, clamp pacing_speed."""
    budget = obs.get("total_budget_remaining", 1000.0)
    allocs = {c: max(0.0, float(data.get("allocations", {}).get(c, 0.0))) for c in CAMPAIGN_NAMES}
    total  = sum(allocs.values())
    if total > budget and total > 0:
        scale  = budget / total
        allocs = {k: round(v * scale, 2) for k, v in allocs.items()}
    mask   = [f for f in data.get("feature_mask", ["I1"]) if f in VALID_FEATURES][:5]
    if not mask:
        mask = ["I1"]
    pacing = float(data.get("pacing_speed", 1.0))
    pacing = max(0.5, min(2.0, pacing))
    return {
        "allocations":       allocs,
        "attribution":       data.get("attribution", "last_click"),
        "feature_mask":      mask,
        "use_capi":          bool(data.get("use_capi", False)),
        "pacing_speed":      pacing,
        "halted_campaigns":  [],
        "legal_reason_code": None,
    }


def extract_json(text):
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    return None


def model_act(obs):
    """Run fine-tuned model inference and return a sanitized action dict."""
    phase       = obs.get("platform_health", "Nominal")
    instruction = PHASE_INSTRUCTIONS.get(phase, PHASE_INSTRUCTIONS["Nominal"])
    prompt      = ALPACA_PROMPT.format(instruction=instruction, input=obs_to_text(obs))
    inputs      = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 200,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    data     = extract_json(response)

    if data is None:   # fallback: equal split of step budget
        budget = obs.get("total_budget_remaining", 0)
        step   = max(1, obs.get("step", 1))
        per    = round(budget / step / 3, 2)
        data   = {"allocations": {c: per for c in CAMPAIGN_NAMES},
                  "attribution": "last_click", "feature_mask": ["I1"],
                  "use_capi": False, "pacing_speed": 1.0}

    return sanitize_action(data, obs)


# API wrappers
def api_get(path):
    r = requests.get(f"{SPACE_URL}{path}", timeout=30)
    r.raise_for_status()
    return r.json()

def api_post(path, body=None):
    r = requests.post(f"{SPACE_URL}{path}", json=body or {}, timeout=60)
    r.raise_for_status()
    return r.json()


# Wake the Space (it may be sleeping)
print("Waking HF Space...")
try:
    h = api_get("/health")
    print(f"Space ready: {h}")
except Exception as e:
    print(f"Health check failed: {e}")
    print("Space may still be loading — wait 30s then re-run this cell")

In [ ]:
# -- 5. Run evaluation ---------------------------------------------------------
results = {}   # task_id -> [score_seed42, score_seed123, score_seed456]

for task_id in EVAL_TASKS:
    name      = TASK_NAMES[task_id]
    max_steps = TASK_STEPS[task_id]
    results[task_id] = []
    print(f"\n{'='*60}")
    print(f"Task {task_id} -- {name}  ({max_steps} steps/episode)")
    print(f"{'='*60}")

    for seed in SEEDS:
        print(f"  seed={seed}  ", end="", flush=True)

        obs   = api_post("/reset", {"task_id": task_id, "seed": seed})
        step  = 0
        done  = False
        rwds  = []

        while not done and step < max_steps:
            action = model_act(obs)
            result = api_post("/step", action)
            obs    = result["observation"]
            rwds.append(result["reward"])
            done   = result["done"]
            step  += 1
            print(".", end="", flush=True)

        grade = api_post("/grader", {"task_id": task_id})
        score = grade.get("score", 0.0)
        results[task_id].append(score)
        avg_r = sum(rwds) / max(1, len(rwds))
        print(f"  score={score:.4f}  steps={step}  avg_reward={avg_r:.3f}")

print("\nAll episodes complete.")

In [ ]:
# -- 6. Results table ----------------------------------------------------------
print("\n" + "="*65)
print(f"{'Task':<28} {'Baseline':>9} {'Fine-tuned':>11} {'Delta':>8}")
print("-"*65)

all_base = []
all_ft   = []

for task_id in EVAL_TASKS:
    name    = TASK_NAMES[task_id]
    base    = BASELINE[task_id]
    scores  = results[task_id]
    avg     = sum(scores) / len(scores)
    delta   = avg - base
    sign    = "+" if delta >= 0 else ""
    seeds_s = " / ".join(f"{s:.3f}" for s in scores)
    all_base.append(base)
    all_ft.append(avg)
    print(f"Task {task_id} {name:<23} {base:>9.4f} {avg:>11.4f}  {sign}{delta:.4f}")
    print(f"  seeds: {seeds_s}")

print("-"*65)
avg_base = sum(all_base) / len(all_base)
avg_ft   = sum(all_ft)   / len(all_ft)
delta    = avg_ft - avg_base
sign     = "+" if delta >= 0 else ""
print(f"{'Average':<28} {avg_base:>9.4f} {avg_ft:>11.4f}  {sign}{delta:.4f}")
print("="*65)

verdict = "BEATS" if avg_ft > avg_base else "is below"
pct     = abs(delta) / avg_base * 100
print(f"\nFine-tuned model {verdict} ExpertBot baseline by {pct:.1f}%")